In [ ]:
import xarray as xr
import rioxarray
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.base import RegressorMixin
from sklearn.metrics import mean_squared_error, explained_variance_score, r2_score
from sklego.meta import ZeroInflatedRegressor
from sklearn.compose import TransformedTargetRegressor
from scipy.special import logit, expit # expit == invlogit
import os
import lightgbm as lgb
import fasttreeshap
import warnings
import logging
import geopandas as gpd
from itertools import product
from tqdm.autonotebook import tqdm

import const

## Load data

In [ ]:
westmort = xr.open_zarr("../data_working/westmort.zarr/").compute().rio.write_crs(const.PROJECTION)
westmort.assign_coords(time=pd.DatetimeIndex(westmort.time))
westmort

## Feature engineering

We are specifically interested in out-year forecasting, so the target variable is each DCA shifted backward by one year.

In [ ]:
mort_vars = list(filter(
    lambda x: x.endswith("mort"),
    list(westmort.variables.keys())
))

target_ds = westmort[mort_vars].shift(time=-1)

Note that the data changes, not the coordinates! For example

In [ ]:
(
    target_ds["doug_fb_mort"].sel(time="2004-01-01").fillna(-1) == 
    westmort["doug_fb_mort"].sel(time="2005-01-01").fillna(-1)
).all()

This means that we can pull data from 2004 in both datasets and it will accurately represent the prediction problem.

In [ ]:
# Rename variables and merge
var_rename_dict = {
    x:x.replace("mort", "target")
    for x in mort_vars
}

target_ds_rename = target_ds.rename(**var_rename_dict)

westmort_merge = xr.combine_by_coords([westmort, target_ds_rename])
westmort_merge

## Identify years with greatest mortality

For exploratory analysis, we should focus on forecasting years with the greatest mortality. Identify the peak year for each agent.

In [ ]:
target_vars = list(filter(lambda x: x.endswith("_target"), westmort_merge.variables.keys()))

idx_of_max_mort = westmort_merge[target_vars].sum(dim=["x", "y"]).argmax(dim="time").to_dataarray()
year_of_max_mort = westmort_merge.time.isel(time=idx_of_max_mort)
year_of_max_mort

## Generating data frames for training

Going from xarray -> dataframe will eat a lot of memory. One way to mitigate this is to call `westmort_merge.sel(...)` to get the years/insects you want and then convert to dataframe.

In [ ]:
COVARIATES = {
    "hydro": ["HT", "P50", "WUE", "rdmax", "gsmax"],
    "topo": ["elev", "heat"],
    "climate": ["tmin", "vpd", "def"]
}

In [ ]:
def make_data_frame(agent: str, valid_year: int, lookback_years: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    agent_ba = f"{agent}_ba"
    agent_mort = f"{agent}_mort"
    agent_target = f"{agent}_target"
    
    cols_to_select = COVARIATES["hydro"] +\
        COVARIATES["topo"] +\
        COVARIATES["climate"] +\
        [agent_ba, agent_mort, agent_target]

    start_year = valid_year - lookback_years
    
    westmort_subset = westmort_merge.sel(time=slice(f"{start_year}-01-01", f"{valid_year}-01-01"))

    # Split into train/validation periods
    train = westmort_subset.isel(time=slice(None, -1)).to_dataframe()
    valid = westmort_subset.isel(time=[-1]).to_dataframe()

    # Only keep pixels with no nans and nonzero host ba
    train = train[train[agent_ba] > 0].dropna()
    valid = valid[valid[agent_ba] > 0].dropna()
    
    return train, valid

In [ ]:
%%time
train, valid = make_data_frame("fir_eng", 2010, 4)

## Model training functions

Here we are replicating the approach in Francis et al. (2025), but with a GBM instead of a simple logistic model. The full structure is:
 - Logit-transform response with bounds from 0-1
 - Hurdle model, with classifier arm doing quantile regression for the median
 - Invlogit-transform predictions

Logit transformation results in +/- Inf if we have zeros or ones in the input data. There are no ones, but we do have a lot of zeros. 

In [ ]:
def safe_logit(x, min=0, max=100, eps=0.1):
    '''
    Calculates logits for data over the range [min-eps, max+eps] to prevent
    +/-Inf when x == min or x == max.

    Set eps=0 to get plain logits.
    '''
    new_min = min - eps
    new_max = max + eps

    x_scale = (x - new_min) / (new_max - new_min)
    return logit(x_scale)

def safe_inv_logit(x, min=0, max=100, eps=0.1):
    '''
    Inverse of safe_logit.
    '''
    x_scale = expit(x)
    new_max = max + eps
    new_min = min - eps
    x = (x_scale * (new_max - new_min)) + new_min
    return x

In [ ]:
x = np.random.uniform(low=0, high=100, size=100)
logits = safe_logit(x)
invlogit = safe_inv_logit(logits)
assert np.allclose(x, invlogit)

In [ ]:
def make_base_estimator(**kwargs) -> lgb.LGBMRegressor:
    return lgb.LGBMRegressor(verbosity=-1, random_state=1234, **kwargs)

def make_zif_estimator(classif_args: dict={}, regressor_args: dict={}) -> ZeroInflatedRegressor:
    return ZeroInflatedRegressor(
        lgb.LGBMClassifier(verbosity=-1, random_state=1234, **classif_args),
        lgb.LGBMRegressor(verbosity=-1, random_state=1234, **regressor_args)
    )

def make_zif_quantile_estimator(classif_args: dict={}, regressor_args: dict={}) -> ZeroInflatedRegressor:
    classif = lgb.LGBMClassifier(verbosity=-1, random_state=1234, **classif_args)
    quantile_regressor = lgb.LGBMRegressor(verbosity=-1, random_state=1234, **regressor_args)

    return ZeroInflatedRegressor(
        classif,
        TransformedTargetRegressor(
            regressor=quantile_regressor,
            func=safe_logit,
            inverse_func=safe_inv_logit
        )
    )

In [ ]:
def split_xy(df: pd.DataFrame, target: str) -> tuple[np.array, np.array]:
    return (
        df.drop(columns=target).to_numpy(),
        df[target].to_numpy()
    )

def get_results(y: np.array, y_hat: np.array) -> dict[str, float]:
    mse = mean_squared_error(y, y_hat)
    nrmse = np.sqrt(mse) / np.std(y)
    r2 = r2_score(y, y_hat)
    ev = explained_variance_score(y, y_hat)
    return {
        "mse": mse,
        "nrmse": nrmse,
        "r2": r2,
        "exp_var": ev,
        "n": y.shape[0]
    }

def train_model(model: RegressorMixin, train: pd.DataFrame, valid: pd.DataFrame, target_var: str, valid_year: int) -> dict:
    if train.shape[0] == 0 or valid.shape[0] == 0:
        # Subsetting resulted in no data
        return {
            "year": start_year,
            "agent": target_var
        }
    
    train_x, train_y = split_xy(train, target_var)
    valid_x, valid_y = split_xy(valid, target_var)

    model.fit(train_x, train_y)

    valid_y_hat = model.predict(valid_x)
    train_y_hat = model.predict(train_x)

    valid_results = get_results(valid_y, valid_y_hat)
    train_results = get_results(train_y, train_y_hat)

    results = {}
    results["year"] = valid_year
    results["agent"] = target_var
    results["valid"] = valid_results
    results["train"] = train_results
    results["model"] = model

    return results

## Parameter sensitivity analysis
To test
 - Lookback period
 - Number of estimators
 - Tree depth
 - Learning rate

In [ ]:
warnings.filterwarnings("ignore", message="X does not have valid feature names")
logging.getLogger().setLevel(logging.CRITICAL)

In [ ]:
def eval_parameter_sensitivity(default_args: dict, param_name: str, param_values: list) -> pd.DataFrame:
    agents = list(const.HOST_DCA_CODES.keys())
    param_space = list(product(param_values, agents))

    results = []

    for (param_value, agent) in tqdm(param_space):
        args = default_args.copy()
        args[param_name] = param_value
        result = eval_single_model(agent, **args)
        result.update(args)
        results.append(result)

    return pd.json_normalize(results)

def eval_single_model(agent: str, lookback_window_size: int, **gbm_args):
    target_var = f"{agent}_target"
    valid_year = year_of_max_mort.sel(variable=target_var).time.dt.year.data
    
    train, valid = make_data_frame(agent, valid_year, lookback_window_size)
    model = make_zif_quantile_estimator(gbm_args, gbm_args) # same args for classifier and regressor

    ret = train_model(
        model, train, valid, target_var, valid_year
    )

    return ret

In [ ]:
def plot_sens_result(df: pd.DataFrame, param_name: str, value_name: str="valid.exp_var"):
    fig, (left, right) = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))

    df.boxplot(column="valid.exp_var", by=param_name, ax=left, vert=False)
    df.boxplot(column="valid.exp_var", by="agent", ax=right, vert=False)

    plt.tight_layout()
    plt.suptitle("")
    left.set_title(f"valid.exp_var by {param_name}")
    right.set_title("valid.exp_var by agent")

    return fig, (left, right)

In [ ]:
default_args = {
    "lookback_window_size": 4,
    "n_estimators": 100,
    "max_depth": -1,
    "learning_rate": 0.1
}

lookback_sens = eval_parameter_sensitivity(
    default_args,
    "lookback_window_size",
    [2, 3, 4, 5, 6]
)

plot_sens_result(lookback_sens, "lookback_window_size")
plt.show()

In [ ]:
n_est_sens = eval_parameter_sensitivity(
    default_args,
    "n_estimators",
    [50, 100, 500, 1000]
)

plot_sens_result(n_est_sens, "n_estimators")
plt.show()

In [ ]:
# Depth is roughly log2(num_leaves)
depth_sens = eval_parameter_sensitivity(
    default_args,
    "num_leaves",
    [8, 16, 32, 64, 128]
)

plot_sens_result(depth_sens, "num_leaves")
plt.show()

In [ ]:
lr_sens = eval_parameter_sensitivity(
    default_args,
    "learning_rate",
    [0.01, 0.05, 0.1, 0.5]
)

plot_sens_result(lr_sens, "learning_rate")
plt.show()

Turns out the default arguments work great ¯\\\_(ツ)\_/¯

In [ ]:
pd.concat([lookback_sens, n_est_sens, depth_sens, lr_sens]).to_csv("../data_working/gbm_param_sensitivity.csv")

## Temporal CV

Use a sliding 5-year window to derive training/validation sets.

In [ ]:
gbm_args = {
    "learning_rate": 0.1,
    "n_estimators": 100,
    "num_leaves": 16,
}

lookback_window_size = 4

year_min = 2001 # early surveys are wacky, ignore them
year_max = westmort_merge.time.max().dt.year.data

valid_years = np.arange(year_min + lookback_window_size, year_max+1)
agents = list(const.HOST_DCA_CODES.keys())

param_space = list(product(valid_years, agents))
print("N iterations:", len(param_space))
print(param_space[0])

In [ ]:
temporal_cv_results = []

for (valid_year, agent) in tqdm(param_space):
    target_var = f"{agent}_target"
    
    train, valid = make_data_frame(agent, valid_year, lookback_window_size)
    if train.shape[0] == 0 or valid.shape[0] == 0:
        continue # ignore years with empty data
        
    model = make_zif_quantile_estimator(gbm_args, gbm_args) # same args for classifier and regressor

    ret = train_model(
        model, train, valid, target_var, valid_year
    )

    temporal_cv_results.append(ret)

temporal_cv_df = pd.json_normalize(temporal_cv_results)

temporal_cv_df.to_csv("../data_working/gbm_temporal_cv.csv")

In [ ]:
fig, ax = plt.subplots()

temporal_cv_df.boxplot("valid.exp_var", by="agent", vert=False, ax=ax)
ax.set_xlim(-1, 1)
plt.show()